# LSTM Encoder for Sentiment Analysis

**Objective:** Build a custom LSTM-based encoder to classify movie reviews from the IMDB dataset as positive or negative.

This notebook walks through the full pipeline:
1. Loading and exploring the dataset
2. Tokenization and dataset preparation
3. Building a custom LSTM encoder from scratch
4. Training and evaluation loops
5. Running the experiment and analyzing results

**Key concepts:** LSTM (Long Short-Term Memory), sequence classification, word embeddings, BERT tokenizer.

## 1. Load the Dataset

We use the **IMDB Dataset** containing 50,000 movie reviews labeled as either *positive* or *negative*. This is a standard benchmark for binary sentiment classification.

In [24]:
import pandas as pd
df = pd.read_csv("IMDB_DATASET.csv")
print(df.head(5))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


## 2. Tokenization and Dataset Class

We leverage the **BERT tokenizer** (`bert-base-uncased`) to convert raw text into token IDs. Each review is:
- **Tokenized** into subword units using BERT's WordPiece vocabulary (~30K tokens)
- **Truncated** or **padded** to a fixed maximum length for uniform batch processing

The custom `IMDBData` class wraps this logic into a PyTorch `Dataset` for seamless integration with `DataLoader`.

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

class IMDBData(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        encoding = self.tokenizer(
            row["review"],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        x = encoding["input_ids"].squeeze(0)
        y = torch.tensor(1 if row["sentiment"] == "positive" else 0, dtype=torch.long)
        return x, y

## 3. Custom LSTM Encoder

The model architecture consists of three layers:

| Layer | Description |
|-------|-------------|
| **Embedding** | Maps token IDs to dense vectors of size `embed_dim` |
| **LSTM** | Processes the sequence and captures temporal dependencies |
| **Linear (FC)** | Maps the final hidden state to 2 output classes (positive / negative) |

We use only the **last hidden state** of the LSTM (`hidden[-1]`) as the sentence-level representation for classification. This is a common approach in sequence encoding where the final state summarizes the entire input.

In [ ]:
import torch.nn as nn

class CustomLSTMEncoder(nn.Module):
    def __init__(self, tokenizer, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)  # 2 classes: negative, positive

    def forward(self, x):
        emb = self.embedding(x)                  # (batch, seq_len, embed_dim)
        _, (hidden, _) = self.lstm(emb)           # hidden: (num_layers, batch, hidden_dim)
        out = self.fc(hidden[-1])                 # (batch, 2)
        return out

## 4. Training Loop

Standard PyTorch training loop: forward pass, compute loss, backpropagate gradients, and update weights. Loss is printed every 100 batches to monitor convergence.

In [27]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * dataloader.batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

## 5. Evaluation Loop

Evaluates the model on the test set with gradients disabled (`torch.no_grad()`). Reports overall **accuracy** and **average loss** across all batches.

In [28]:
def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## 6. Experiment Setup and Training

**Hyperparameters:**

| Parameter | Value |
|-----------|-------|
| Max sequence length | 200 |
| Embedding dimension | 128 |
| Hidden dimension | 128 |
| LSTM layers | 1 |
| Learning rate | 1e-3 |
| Batch size | 64 |
| Epochs | 5 |
| Optimizer | Adam |
| Loss function | CrossEntropyLoss |

The dataset is split **80/20** into training and test sets using `random_split`.

In [ ]:
MAX_LEN = 200

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.save_pretrained('./mi_tokenizer_bert_local')

print(f"Vocabulary: {tokenizer.vocab_size} tokens")
print(f"Example: {tokenizer.tokenize('This movie is great!')}")

# Train/test split with torch
train_size = int(len(df) * 0.8)
test_size = len(df) - train_size

full_dataset = IMDBData(df, tokenizer, MAX_LEN)
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)

### Model Initialization and Training

We instantiate the LSTM encoder and train it for 5 epochs using the **Adam** optimizer and **CrossEntropyLoss**.

In [ ]:
model = CustomLSTMEncoder(tokenizer, embed_dim=128, hidden_dim=128, num_layers=1)

learning_rate = 1e-3
epochs = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

## 7. Results and Conclusions

The LSTM encoder achieves **~83.9% accuracy** on the test set after just 5 epochs of training, demonstrating that even a simple single-layer LSTM can effectively capture sentiment patterns in text.

**Key observations:**
- The model starts near random chance (~53%) and steadily improves across epochs
- The biggest accuracy jump occurs between epochs 3 and 5 (70.5% to 83.9%)
- Both training loss and test loss decrease consistently, indicating no significant overfitting

**Potential improvements:**
- Increase the number of LSTM layers or hidden dimensions
- Use bidirectional LSTM to capture context from both directions
- Add dropout for regularization
- Train for more epochs with a learning rate scheduler
- Experiment with pre-trained embeddings (e.g., GloVe, Word2Vec)